<a href="https://colab.research.google.com/github/kanebako-s/olist-data-analysis/blob/main/Kaggle_Olist_analysis_v2.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# --- ライブラリのインポート ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# --- グラフの見た目を整える ---
sns.set(style="whitegrid") # グラフにグリッド（網掛け）を出して見やすくする
plt.rcParams["figure.figsize"] = (10, 6) # グラフの基本サイズを少し大きくする

# --- データの表示設定 ---
pd.options.display.max_columns = None # 列数が多い時、省略（...）せずに全部表示する
pd.set_option('display.float_format', '{:.2f}'.format) # 小数点以下を2桁に固定して見やすくする

# --- 日本語化（重要！） ---
!pip install japanize-matplotlib
import japanize_matplotlib # これでグラフの日本語が「□」にならずに済む

# --- Worldcloudのインストール ---
!pip install wordcloud
from wordcloud import WordCloud

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 36.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for japanize-matplotlib: filename=japanize_matplotlib-1.1.3-py3-none-any.whl size=4120257 sha256=99e193d76ad5d3a0b54de75fab7aa5cb6470c5d7fdc5f4a5f59430c106da5a19
  Stored in directory: /root/.cache/pip/wheels/c1/f7/9b/418f19a7b9340fc16e071e89efc379aca68d40238b258df53d
Successfully built japanize-matplotlib


# Executive Summary

# Data Preparation

** データの統合と整合性 **

 ここでは、分析の土台となるデータセットの作成を行います。


* データの整合性 : 分析に正確性を期すために1注文=1行になるように集計を行いました。単純な結合の場合11万9千件にまで増殖してしまいますが、重複を排除することで信頼性のあるデータにしています。

* 変数の作成 : リピート分析を主眼に置いているため注文合計金額（total_item_price）を作成することでリピート率の改善が売り上げにどれだけの影響が出るのか正しく評価できるようにしました。

In [3]:
from pathlib import Path

# ファイルの読み込み。
BASE_DIR = Path('/content/drive/MyDrive/Olist_Dateanalist_Learning') # ファイルの読み込みが自分専用になっている。あとで直しやすいようにしている。

df_customers = pd.read_csv(BASE_DIR / 'olist_customers_dataset.csv')
df_orders = pd.read_csv(BASE_DIR / 'olist_orders_dataset.csv')
df_order_reviews = pd.read_csv(BASE_DIR / 'olist_order_reviews_dataset.csv')
df_order_items = pd.read_csv(BASE_DIR / 'olist_order_items_dataset.csv')
df_products = pd.read_csv(BASE_DIR / 'olist_products_dataset.csv')
df_order_payments = pd.read_csv(BASE_DIR / 'olist_order_payments_dataset.csv')

# テーブルの結合。
# 分析に必要な列だけを定義。
customer_cols = ['customer_id', 'customer_unique_id']
item_cols = ['order_id', 'product_id', 'seller_id', 'price']
product_cols = ['product_id', 'product_photos_qty', 'product_description_lenght']
timestamps_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'review_creation_date', 'review_answer_timestamp']

# 集計で重複が発生しないように注文ごとにまとめる。
# マージする前に、注文ごとの合計支払額にまとめてしまう。
df_payments_sum = df_order_payments.groupby('order_id')['payment_value'].sum().reset_index()

# df_order_items を注文単位で集計。
df_items_sum = df_order_items.groupby('order_id').agg({
    'product_id': 'first',  # 最も高い商品のIDを1つだけ取る（これでdf_productsと繋がる！）
    'seller_id': 'first',   # その商品のセラーを取る
    'price': 'sum',         # 価格は「合計金額」として残す
}).reset_index()

# カラム名を分かりやすく変えておく。
df_items_sum = df_items_sum.rename(columns={'price': 'total_item_price'})

# レビューを注文IDごとに1件に絞る（最新の投稿時間から最終的なレビューコメントを代表値として残す）
df_reviews_unique = (
    df_order_reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates('order_id')
)

# メソッドチェーンでマージ。
df_final = (
    df_orders
    .merge(df_customers[customer_cols], on='customer_id', how='left')
    .merge(df_items_sum, on='order_id', how='left')
    .merge(df_reviews_unique, on='order_id', how='left')
    .merge(df_products[product_cols], on='product_id', how='left')
    .merge(df_payments_sum, on='order_id', how='left')
)

# ループで一気に日付型に変換。
for col in timestamps_cols:
    df_final[col] = pd.to_datetime(df_final[col])

# df_final.shape
df_orders.shape

(99441, 8)

結果: 行数が一致したことを確認。

** データのノイズの排除 **

欠損値を確認してノイズとなるデータを特定する。

* 注文の承認時間（order_approved_at）が160件欠損していた。これは顧客が注文を承認する前にキャンセルしたということだから分析のノイズになり得るので除外します。

* 商品情報で写真掲載枚数（product_photos_qty）と商品説明（product_description_lenght）が2191件欠損していた。なんの情報もない商品の注文が2191件あったということは「思っていたのとは違った」という期待感のズレが発生している可能性があります。

  **

* （product_id, seller_id, total_item_price）の3つが775件欠損していた。これは注文承認時間の欠損より多く、注文承認はされたけどその後取引破棄になったものが600件くらいある事がわかります。

  ** 注文承認が降りた後取引破棄になったデータ **

  * 利用不可（unavailable）が603件もあることから、注文が承認されて届くのを待っていたら「在庫ないからキャンセル」と言われた。そんなシチュエーションが想像できます。

In [5]:
# 注文承認済み(not null) なのに、商品がない(null) 注文のステータスは？
print(df_final[df_final['order_approved_at'].notnull() & df_final['product_id'].isnull()]['order_status'].value_counts())

order_status
unavailable    603
canceled        23
invoiced         2
shipped          1
Name: count, dtype: int64


In [4]:
print(df_final.isnull().sum())

order_id                             0
customer_id                          0
order_status                         0
order_purchase_timestamp             0
order_approved_at                  160
order_delivered_carrier_date      1783
order_delivered_customer_date     2965
order_estimated_delivery_date        0
customer_unique_id                   0
product_id                         775
seller_id                          775
total_item_price                   775
review_id                          768
review_score                       768
review_comment_title             87889
review_comment_message           58667
review_creation_date               768
review_answer_timestamp            768
product_photos_qty                2191
product_description_lenght        2191
payment_value                        1
dtype: int64
